<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Java Setup (needed for pyspark)
!apt-get install -y openjdk-17-jre 2>/dev/null > /dev/null

In [ ]:
#@title Download 1% sample
!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

In [ ]:
#@title Dataset Schema
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.master('local[*]').appName('taxis').getOrCreate()

try :
    data = spark.read.csv('taxi_rides_1pc.csv.gz', sep =',', header=True, inferSchema=True)

    data.printSchema()

except Exception as err:
    print(err)

In [ ]:
#@title Clean the data
from pyspark.sql.functions import col, isnan, when, floor

# Filter out rows with null or invalid coordinate values
cleaned_data = data.filter(
    col("pickup_latitude").isNotNull() & ~isnan(col("pickup_latitude")) &
    col("pickup_longitude").isNotNull() & ~isnan(col("pickup_longitude")) &
    col("dropoff_latitude").isNotNull() & ~isnan(col("dropoff_latitude")) &
    col("dropoff_longitude").isNotNull() & ~isnan(col("dropoff_longitude"))
)

# Longitude and latitude from the upper left corner of the grid
MIN_LON = -74.916578
MAX_LAT = 41.47718278

# Longitude and latitude that correspond to a shift in 500 meters
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

# Convert coordinates (latitude,longitude) to grids (x,y)
data_with_grid_coords = cleaned_data \
    .withColumn("pickup_grid_x", floor((MAX_LAT - col("pickup_latitude")) / LAT_DELTA)) \
    .withColumn("pickup_grid_y", floor((col("pickup_longitude") - MIN_LON) / LON_DELTA)) \
    .withColumn("dropoff_grid_x", floor((MAX_LAT - col("dropoff_latitude")) / LAT_DELTA)) \
    .withColumn("dropoff_grid_y", floor((col("dropoff_longitude") - MIN_LON) / LON_DELTA))

# Filter the DataFrame to include only trips within the 300x300 grid for both pickup and dropoff
filtered_data = data_with_grid_coords.filter(
    (col("pickup_grid_x") > 0) & (col("pickup_grid_x") < 300) &
    (col("pickup_grid_y") > 0) & (col("pickup_grid_y") < 300) &
    (col("dropoff_grid_x") > 0) & (col("dropoff_grid_x") < 300) &
    (col("dropoff_grid_y") > 0) & (col("dropoff_grid_y") < 300)
)

In [ ]:
#@title Calculate trip_profit
final_data = filtered_data.withColumn("trip_profit", col("fare_amount") + col("tip_amount"))

In [ ]:
#@title Display data reduction results and final data schema
print("Original DataFrame count:", data.count())
print("Cleaned DataFrame count (after removing null coords):", cleaned_data.count())
print("DataFrame with grid coordinates, filtered and profit count:", final_data.count())
final_data.printSchema()

#Q1: Are some areas more profitable than others?

The profitability of a pickup area is determined by dividing the area profit by the number of empty taxis in that area within the last 15 minutes.

1. The area profit is computed by calculating the average profit (fare+tip) for trips that started in the area and ended within 15 minutes from the pickup time.

2. The number of empty taxis in the area is the sum of taxis that had a drop-off location in that area less than 30 minutes ago and had no following pickup yet.

3. Calculate the area profitability by dividing the average area profit by the number of empty taxis in that area.

In [ ]:
#@title 1. Calculate average profit per area
from pyspark.sql.functions import avg, col, unix_timestamp

# Calculate trip duration in minutes
# (unix_timestamp returns seconds, so divide by 60 for minutes)
filtered_for_profit_calculation = final_data.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60
)

# Filter to include only trips where the duration is 15 minutes or less
filtered_for_profit_calculation = filtered_for_profit_calculation.filter(col("trip_duration_minutes") <= 15)

# Group by pickup grid coordinates and calculate the average trip_profit for the filtered data
avg_profit_per_pickup_area = filtered_for_profit_calculation.groupBy("pickup_grid_x", "pickup_grid_y").agg(avg("trip_profit").alias("avg_trip_profit"))

# Display the schema and first few rows of the resulting DataFrame
print("Schema of avg_profit_per_pickup_area:")
avg_profit_per_pickup_area.printSchema()
print("\nFirst 10 rows of avg_profit_per_pickup_area:")
avg_profit_per_pickup_area.show(10)

In [ ]:
#@title 2. Determine number of empty taxis per dropoff area
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, count, col, datediff, unix_timestamp, when, lit

# Define the window specification: partition by medallion, order by pickup_datetime
window_spec = Window.partitionBy("medallion").orderBy("pickup_datetime")

# Add next_pickup_datetime
data_with_next_pickup = final_data.withColumn(
    "next_pickup_datetime",
    lead(col("pickup_datetime"), 1).over(window_spec)
)

# Calculate time_to_next_pickup_minutes
# Convert timestamps to Unix timestamps (seconds) for accurate difference calculation
data_with_next_pickup = data_with_next_pickup.withColumn(
    "time_to_next_pickup_minutes",
    (unix_timestamp(col("next_pickup_datetime")) - unix_timestamp(col("dropoff_datetime"))) / 60
)

# Create is_empty_taxi status
# Flag data for which the next_pickup_datetime is null (last trip) or the time_to_next_pickup_minutes is > 30
final_data_with_empty_status = data_with_next_pickup.withColumn(
    "is_empty_taxi",
    when(
        (col("next_pickup_datetime").isNull()) | (col("time_to_next_pickup_minutes") > 30),
        True
    ).otherwise(False)
)

# Display schema and first few rows
print("Schema of DataFrame with empty taxi status:")
final_data_with_empty_status.printSchema()
print("\nFirst 10 rows of DataFrame with empty taxi status (showing relevant columns):")
final_data_with_empty_status.select(
    "medallion", "pickup_datetime", "dropoff_datetime",
    "next_pickup_datetime", "time_to_next_pickup_minutes",
    "is_empty_taxi"
).orderBy("medallion", "pickup_datetime").show(10, truncate=False)

# Calculate the count of empty taxi instances per dropoff grid area
empty_taxi_count_per_dropoff_area = final_data_with_empty_status.filter(
    col("is_empty_taxi") == True
).groupBy("dropoff_grid_x", "dropoff_grid_y") \
 .agg(count("medallion").alias("empty_taxi_count"))

# Display the schema and first few rows of the resulting DataFrame
print("Schema of empty_taxi_count_per_dropoff_area:")
empty_taxi_count_per_dropoff_area.printSchema()
print("\nFirst 10 rows of empty_taxi_count_per_dropoff_area:")
empty_taxi_count_per_dropoff_area.show(10)

In [ ]:
#@title 3. Calculate area profitability
from pyspark.sql.functions import col, when, desc

# Join the two DataFrames on matching grid coordinates
area_profitability = avg_profit_per_pickup_area.join(
    empty_taxi_count_per_dropoff_area,
    (avg_profit_per_pickup_area["pickup_grid_x"] == empty_taxi_count_per_dropoff_area["dropoff_grid_x"]) &
    (avg_profit_per_pickup_area["pickup_grid_y"] == empty_taxi_count_per_dropoff_area["dropoff_grid_y"]),
    "inner" # Use inner join to only consider areas present in both
)

# Calculate the profitability ratio, handling division by zero
area_profitability = area_profitability.withColumn(
    "profitability",
    when(col("empty_taxi_count") > 0,
         col("avg_trip_profit") / col("empty_taxi_count"))
    .otherwise(0.0) # Assign 0.0 or another suitable value if empty_taxi_count is zero
)

# Order the output by profitability ratio in descending order
area_profitability = area_profitability.orderBy(desc("profitability"))

# Display the schema and first few rows of the resulting DataFrame
print("Schema of area_profitability:")
area_profitability.printSchema()
print("\nFirst 10 rows of area_profitability:")
area_profitability.show(10)

In [ ]:
#@title Show a heat map of the profitability data to highlight the most profitable areas.
import matplotlib.pyplot as plt

# Convert the Spark DataFrame to a Pandas DataFrame for visualization
profitability_df_pd = area_profitability.toPandas()

# Create a scatter plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(
    profitability_df_pd['pickup_grid_x'],
    profitability_df_pd['pickup_grid_y'],
    c=profitability_df_pd['profitability'], # Color by profitability
    cmap='viridis', # Colormap
    s=40, # Size of the points
    alpha=0.8 # Transparency of the points
)

plt.colorbar(scatter, label='Area Profitability')
plt.xlabel('Pickup Grid X')
plt.ylabel('Pickup Grid Y')
plt.title('Area Profitability Heatmap')
plt.grid(True)
plt.show()

#Obervations
Observing the heat map of the average profitability we can conclude that there the majority of the areas do not generate much profit while some of the areas generate high levels of profit.

To obtain further insight about the distribution of the area profitability data we plot the histogram of the top 100 most profitable areas and the boxplot of the trip profits.

In [ ]:
#@title Show top 100 most profitable trips
import matplotlib.pyplot as plt
import seaborn as sns

# Get the top 10 most profitable areas
top_100_profitability = area_profitability.limit(100).toPandas()

# Create a combined grid coordinate string for better labeling
top_100_profitability['grid_coords'] = top_100_profitability['pickup_grid_x'].astype(str) + ',' + top_100_profitability['pickup_grid_y'].astype(str)

# Create the bar chart
plt.figure(figsize=(14, 7))
sns.barplot(x='grid_coords', y='profitability', data=top_100_profitability, palette='viridis', hue='grid_coords', legend=False)

plt.xlabel('Pickup Grid Coordinates (X,Y)')
plt.ylabel('Profitability Ratio')
plt.title('Top 100 Most Profitable Areas')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

The histogram of the area profitability data confirms our observation from the heat map, showing that the distribution is right-skewed with the majority of the trips generating low profits and only some of the trips being substantially profitable.

In [ ]:
#@title Skewness of trip profit distribution
import matplotlib.pyplot as plt
import seaborn as sns

# Extract the 'trip_profit' column and convert it to a Pandas Series for plotting
trip_profit_pd = final_data.select('trip_profit').toPandas()['trip_profit']

plt.figure(figsize=(10, 6))
sns.histplot(trip_profit_pd, bins=50)
plt.title('Distribution of Trip Profit')
plt.xlabel('Trip Profit')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

# To get a more detailed view of the central tendency and outliers, let's also look at a box plot.
plt.figure(figsize=(10, 4))
sns.boxplot(x=trip_profit_pd)
plt.title('Box Plot of Trip Profit')
plt.xlabel('Trip Profit')
plt.grid(axis='x', alpha=0.75)
plt.show()

The box plot further confirms the right-skewness and highlights the presence of numerous outliers on the higher end, meaning a few trips generate exceptionally high profits compared to the majority.

#Q2: Is there any dependence of the trip profit on the trip distance?



In [ ]:

#@title Create the distance intervals
from pyspark.sql.functions import col, when

# Define distance bins for trip_distance
final_data_with_bins = final_data.withColumn(
    "distance_bin",
    when((col("trip_distance") >= 0) & (col("trip_distance") < 1), "0-1 miles")
    .when((col("trip_distance") >= 1) & (col("trip_distance") < 2), "1-2 miles")
    .when((col("trip_distance") >= 2) & (col("trip_distance") < 5), "2-5 miles")
    .when((col("trip_distance") >= 5) & (col("trip_distance") < 10), "5-10 miles")
    .otherwise(">10 miles")
)

print("Schema of final_data_with_bins:")
final_data_with_bins.printSchema()
print("First 10 rows of final_data_with_bins (showing distance_bin):")
final_data_with_bins.select("trip_distance", "distance_bin").show(10)

In [ ]:
#@title Group the trip profits by the defined distance intervals
from pyspark.sql.functions import avg
import pandas as pd

# Group by distance_bin and calculate the average trip_profit
avg_profit_by_distance = final_data_with_bins.groupBy("distance_bin").agg(avg("trip_profit").alias("average_trip_profit"))

# Define the order for distance bins for better visualization
distance_bin_order = ["0-1 miles", "1-2 miles", "2-5 miles", "5-10 miles", ">10 miles"]

# Convert to Pandas DataFrame and reorder the bins
avg_profit_by_distance_pd = avg_profit_by_distance.toPandas()
avg_profit_by_distance_pd['distance_bin'] = pd.Categorical(avg_profit_by_distance_pd['distance_bin'], categories=distance_bin_order, ordered=True)
avg_profit_by_distance_pd = avg_profit_by_distance_pd.sort_values('distance_bin')

print("Average profit by distance bin (Pandas DataFrame):")
print(avg_profit_by_distance_pd)

In [ ]:
#@title Visualize the trip profits as a funtion of the trips distance
import matplotlib.pyplot as plt
import seaborn as sns

# Create a bar plot to visualize average trip profit by distance bin
plt.figure(figsize=(10, 6))
sns.barplot(x='distance_bin', y='average_trip_profit', data=avg_profit_by_distance_pd, palette='viridis',hue='distance_bin', legend=False)

plt.title('Average Trip Profit by Distance Bin')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Average Trip Profit')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

As can be observed, there is a clear positive correlation between the trip distance and the trip profit. Longer trips consistently yield higher profits, with trips over 10 miles being the most profitable.

#Q3: Is there any dependence of the trip profit on the time of day?

In [ ]:
#@title Create time-of-day intervals
from pyspark.sql.functions import col, hour, when

# Extract the hour from 'pickup_datetime'
final_data_with_hour = final_data.withColumn("pickup_hour", hour(col("pickup_datetime")))

# Create time_of_day_bin based on pickup_hour
final_data_with_time_bins = final_data_with_hour.withColumn(
    "time_of_day_bin",
    when((col("pickup_hour") >= 6) & (col("pickup_hour") <= 11), "Morning (6-11)")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") <= 17), "Afternoon (12-17)")
    .when((col("pickup_hour") >= 18) & (col("pickup_hour") <= 23), "Evening (18-23)")
    .otherwise("Night (0-5)")
)

print("Schema of final_data_with_time_bins:")
final_data_with_time_bins.printSchema()
print("First 10 rows of final_data_with_time_bins (showing pickup_datetime, pickup_hour, time_of_day_bin):")
final_data_with_time_bins.select("pickup_datetime", "pickup_hour", "time_of_day_bin").show(10)

In [ ]:
#@title Group the trip profits by the time-of-day intervals
from pyspark.sql.functions import avg
import pandas as pd

# Group by time_of_day_bin and calculate the average trip_profit
avg_profit_by_time_of_day = final_data_with_time_bins.groupBy("time_of_day_bin").agg(avg("trip_profit").alias("average_trip_profit"))

# Define the order for time of day bins for better visualization
time_of_day_bin_order = ["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"]

# Convert to Pandas DataFrame and reorder the bins
avg_profit_by_time_of_day_pd = avg_profit_by_time_of_day.toPandas()
avg_profit_by_time_of_day_pd['time_of_day_bin'] = pd.Categorical(avg_profit_by_time_of_day_pd['time_of_day_bin'], categories=time_of_day_bin_order, ordered=True)
avg_profit_by_time_of_day_pd = avg_profit_by_time_of_day_pd.sort_values('time_of_day_bin')

print("Average profit by time of day bin (Pandas DataFrame):")
print(avg_profit_by_time_of_day_pd)

In [ ]:
#@title Visualize the trip profits as a function of the time-of-day
import matplotlib.pyplot as plt
import seaborn as sns

# Create a bar plot to visualize average trip profit by time of day bin
plt.figure(figsize=(10, 6))
sns.barplot(x='time_of_day_bin', y='average_trip_profit', data=avg_profit_by_time_of_day_pd, palette='viridis', hue='time_of_day_bin', legend=False)

plt.title('Average Trip Profit by Time of Day')
plt.xlabel('Time of Day')
plt.ylabel('Average Trip Profit')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

As can be observed, the trip profitability varies slightly with the time-of-day.

The Night (0-5 hours) trips show the highest average profit, closely followed by the Afternoon (12-17 hours). Morning (6-11 hours) and Evening (18-23 hours) evidence slightly lower profits.

##Summary
The key insights gained from the deeper profitability analysis are as follows:
*   **Trip Distance:** There is a clear positive correlation between trip distance and average trip profit. Longer trips consistently yield higher profits, with trips over 10 miles being the most profitable.
*   **Time of Day:** Profitability varies slightly by time of day, with "Night (0-5)" hours generally showing the highest average trip profit.

### Data Analysis Key Findings
*   **Profitability by Trip Distance:** Longer trips are significantly more profitable. Trips over 10 miles yielded the highest average profit at approximately \$49.04, compared to approximately \$6.43 for trips between 0-1 miles.
*   **Profitability by Time of Day:** "Night (0-5)" hours showed the highest average profit (approximately \$14.53), closely followed by "Afternoon (12-17)" at approximately \$14.01. "Morning (6-11)" and "Evening (18-23)" had slightly lower average profits (around \$13.34 and \$13.33, respectively).
